In [5]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
!pip install --no-deps --upgrade timm
import importlib.metadata as _md
try: _tv = tuple(int(x) for x in _md.version("torch").split("+")[0].split(".")[:2])
except Exception: _tv = (2, 11)
_torchcodec = {(2, 11): "torchcodec>=0.11.0", (2, 10): "torchcodec==0.10.*", (2, 9): "torchcodec==0.9.*", (2, 8): "torchcodec==0.7.*", (2, 7): "torchcodec==0.5.*"}.get(_tv, "torchcodec>=0.12.0" if _tv >= (2, 12) else "torchcodec")
!pip install "{_torchcodec}"
import torch; torch._dynamo.config.recompile_limit = 64

In [6]:
from unsloth import FastModel
import torch, json, random, re, os, glob, zipfile, urllib.request
import numpy as np, pandas as pd

SEED = 3407
random.seed(SEED); np.random.seed(SEED)

PROMPT        = "Generate synthetic data."
N_VARIANTS    = 8
MAX_SEQ_LEN   = 1024
MAX_EX_TOKENS = 950
N_SYNTH       = 200
HOLDOUT_FRAC  = 0.15
DX_CAP, PX_CAP, LAB_CAP, MED_CAP = 5, 3, 8, 6

print(torch.cuda.get_device_name(0))

Tesla T4


In [7]:
URL = "https://physionet.org/static/published-projects/mimic-iv-demo/mimic-iv-clinical-database-demo-2.2.zip"
if not glob.glob("mimic-iv*/hosp"):
    urllib.request.urlretrieve(URL, "mimic.zip")
    zipfile.ZipFile("mimic.zip").extractall(".")
BASE = [d for d in glob.glob("mimic-iv*") if os.path.isdir(d)][0] + "/"
H, I = BASE + "hosp/", BASE + "icu/"

def T(folder, name):
    for ext in (".csv.gz", ".csv"):
        p = folder + name + ext
        if os.path.exists(p):
            return pd.read_csv(p, low_memory=False, dtype={"icd_code": str})   #
    raise FileNotFoundError(folder + name)

patients = T(H, "patients");        adm   = T(H, "admissions")
dx       = T(H, "diagnoses_icd");   d_dx  = T(H, "d_icd_diagnoses")
px       = T(H, "procedures_icd");  d_px  = T(H, "d_icd_procedures")
labs     = T(H, "labevents");       d_lab = T(H, "d_labitems")
rx       = T(H, "prescriptions");   icu   = T(I, "icustays")

print("admissions:", len(adm), "| unique hadm_id:", adm.hadm_id.nunique(), "| patients:", len(patients))
print("d_labitems columns:", list(d_lab.columns))

admissions: 275 | unique hadm_id: 275 | patients: 100
d_labitems columns: ['itemid', 'label', 'fluid', 'category']


In [8]:
adm = adm.drop_duplicates("hadm_id").merge(
    patients[["subject_id", "gender", "anchor_age", "anchor_year"]], on="subject_id", how="left")
adm["admittime"] = pd.to_datetime(adm["admittime"]); adm["dischtime"] = pd.to_datetime(adm["dischtime"])
adm["age"] = (adm.anchor_age + adm.admittime.dt.year - adm.anchor_year).clip(18, 110)
adm["los_days"] = ((adm.dischtime - adm.admittime).dt.total_seconds() / 86400).round(1)

dxt = dx.merge(d_dx, on=["icd_code", "icd_version"], how="left").dropna(subset=["long_title"]).sort_values(["hadm_id", "seq_num"])
pxt = px.merge(d_px, on=["icd_code", "icd_version"], how="left").dropna(subset=["long_title"]).sort_values(["hadm_id", "seq_num"])
dx_g = {k: g for k, g in dxt.groupby("hadm_id")}
px_g = {k: g for k, g in pxt.groupby("hadm_id")}

if "fluid" in d_lab.columns:
    lab_items = d_lab[d_lab.fluid == "Blood"][["itemid", "label"]]
else:
    lab_items = d_lab[["itemid", "label"]]
lb = labs[labs.hadm_id.notna() & labs.valuenum.notna()][["hadm_id", "itemid", "charttime", "valuenum", "valueuom", "flag"]].copy()
lb["hadm_id"] = lb.hadm_id.astype(int)
lb = lb.merge(lab_items, on="itemid", how="inner")
lb = lb.sort_values("charttime").drop_duplicates(["hadm_id", "label"], keep="first")
TOP_LABS = lb.groupby("label").hadm_id.nunique().sort_values(ascending=False).head(25).index.tolist()
LAB_ORDER = {l: i for i, l in enumerate(TOP_LABS)}
lb = lb[lb.label.isin(TOP_LABS)].copy()
lb["ord"] = lb.label.map(LAB_ORDER)
lb = lb.sort_values(["hadm_id", "ord"])
lb_g = {k: g for k, g in lb.groupby("hadm_id")}
print("Top labs:", TOP_LABS)

rxd = rx.dropna(subset=["drug"]).drop_duplicates(["hadm_id", "drug"])
rx_g = {k: sorted(g.drug.astype(str).tolist()) for k, g in rxd.groupby("hadm_id")}


icu1 = icu.sort_values("intime").drop_duplicates("hadm_id", keep="first").set_index("hadm_id")
icu_d = icu1.to_dict("index")

def nn(v):
    return None if pd.isna(v) else v

def code_list(g, cap):
    return [{"icd_code": str(r.icd_code), "icd_version": int(r.icd_version), "title": str(r.long_title)}
            for r in g.head(cap).itertuples()]

full_records = []
for a in adm.to_dict("records"):
    h = a["hadm_id"]
    d, p, l, m = dx_g.get(h), px_g.get(h), lb_g.get(h), rx_g.get(h, [])
    rec = {
        "gender": a["gender"], "age": int(a["age"]), "race": a["race"],
        "insurance": nn(a["insurance"]), "language": nn(a["language"]), "marital_status": nn(a["marital_status"]),
        "admission_type": a["admission_type"], "admission_location": nn(a["admission_location"]),
        "discharge_location": nn(a["discharge_location"]),
        "los_days": float(a["los_days"]), "died_in_hospital": int(a["hospital_expire_flag"]),
        "icu": ({"first_careunit": icu_d[h]["first_careunit"], "last_careunit": icu_d[h]["last_careunit"],
                 "icu_los_days": round(float(icu_d[h]["los"]), 1)} if h in icu_d else None),
        "num_diagnoses": 0 if d is None else len(d),
        "num_procedures": 0 if p is None else len(p),
        "num_medications": len(m),
        "diagnoses": [] if d is None else code_list(d, DX_CAP),
        "procedures": [] if p is None else code_list(p, PX_CAP),
        "labs": [] if l is None else [{"test": r.label, "value": round(float(r.valuenum), 2),
                                       "unit": nn(r.valueuom), "flag": "abnormal" if pd.notna(r.flag) else "normal"}
                                      for r in l.itertuples()],
        "medications": m,
    }
    full_records.append((int(a["subject_id"]), rec))

print(len(full_records), "admission records")
print("with ICU:", round(np.mean([r["icu"] is not None for _, r in full_records]), 3),
      "| died:", round(np.mean([r["died_in_hospital"] for _, r in full_records]), 3),
      "| mean num_diagnoses:", round(np.mean([r["num_diagnoses"] for _, r in full_records]), 1))
ex = dict(full_records[0][1]); ex["labs"] = ex["labs"][:3]; ex["medications"] = ex["medications"][:5]
print(json.dumps(ex, ensure_ascii=False, indent=1)[:2500])

Top labs: ['Sodium', 'Creatinine', 'Chloride', 'Potassium', 'Urea Nitrogen', 'Platelet Count', 'Hematocrit', 'Glucose', 'Anion Gap', 'Bicarbonate', 'Hemoglobin', 'RDW', 'MCHC', 'MCH', 'MCV', 'Red Blood Cells', 'White Blood Cells', 'Magnesium', 'Phosphate', 'Calcium, Total', 'INR(PT)', 'PT', 'PTT', 'Bilirubin, Total', 'Asparate Aminotransferase (AST)']
275 admission records
with ICU: 0.465 | died: 0.055 | mean num_diagnoses: 16.4
{
 "gender": "M",
 "age": 47,
 "race": "BLACK/CAPE VERDEAN",
 "insurance": "Medicaid",
 "language": "ENGLISH",
 "marital_status": "SINGLE",
 "admission_type": "URGENT",
 "admission_location": "TRANSFER FROM HOSPITAL",
 "discharge_location": "SKILLED NURSING FACILITY",
 "los_days": 9.0,
 "died_in_hospital": 0,
 "icu": {
  "first_careunit": "Coronary Care Unit (CCU)",
  "last_careunit": "Medical Intensive Care Unit (MICU)",
  "icu_los_days": 5.0
 },
 "num_diagnoses": 26,
 "num_procedures": 5,
 "num_medications": 65,
 "diagnoses": [
  {
   "icd_code": "03842",
   

In [9]:
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it", dtype = None,
    max_seq_length = MAX_SEQ_LEN, load_in_4bit = False, full_finetuning = False,
)
model = FastModel.get_peft_model(
    model, finetune_vision_layers = False, finetune_language_layers = True,
    finetune_attention_modules = True, finetune_mlp_modules = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = SEED,
)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")
tok = getattr(tokenizer, "tokenizer", tokenizer)

subjects = sorted({s for s, _ in full_records})
random.Random(SEED).shuffle(subjects)
hold = set(subjects[: int(HOLDOUT_FRAC * len(subjects))])
train_base = [r for s, r in full_records if s not in hold]
hold_base  = [r for s, r in full_records if s in hold]
print("train admissions:", len(train_base), "| holdout admissions:", len(hold_base))

def make_view(rec):
    r = dict(rec)
    labs_ = random.sample(rec["labs"], min(LAB_CAP, len(rec["labs"])))
    labs_.sort(key=lambda x: LAB_ORDER[x["test"]])
    r["labs"] = labs_
    r["medications"] = sorted(random.sample(rec["medications"], min(MED_CAP, len(rec["medications"]))))
    return r

def dumps(r): return json.dumps(r, ensure_ascii=False, separators=(",", ":"))

def chat_text(rec):
    convo = [{"role": "user", "content": PROMPT}, {"role": "assistant", "content": dumps(rec)}]
    return tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)

train_views = [make_view(r) for r in train_base for _ in range(N_VARIANTS)]
texts = [chat_text(r) for r in train_views]
lens = [len(x) for x in tok(texts, add_special_tokens=False)["input_ids"]]
print("tokens mean/max:", int(np.mean(lens)), max(lens))

keep = [i for i, l in enumerate(lens) if l <= MAX_EX_TOKENS]
print(f"dropped (too long): {len(lens) - len(keep)} of {len(lens)}")
kept_lens = [lens[i] for i in keep]
GEN_TOKENS = max(kept_lens) + 30

from datasets import Dataset
dataset = Dataset.from_dict({"text": [texts[i] for i in keep]}).shuffle(seed=SEED)
print(len(dataset), "training examples | GEN_TOKENS =", GEN_TOKENS)
print(dataset[0]["text"][:700])

==((====))==  Unsloth 2026.10.11: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

train admissions: 249 | holdout admissions: 26
tokens mean/max: 529 689
dropped (too long): 0 of 1992
1992 training examples | GEN_TOKENS = 719
<bos><|turn>user
Generate synthetic data.<turn|>
<|turn>model
{"gender":"F","age":53,"race":"WHITE","insurance":"Medicare","language":"ENGLISH","marital_status":"SINGLE","admission_type":"EW EMER.","admission_location":"EMERGENCY ROOM","discharge_location":"HOME HEALTH CARE","los_days":6.9,"died_in_hospital":0,"icu":null,"num_diagnoses":14,"num_procedures":3,"num_medications":34,"diagnoses":[{"icd_code":"00845","icd_version":9,"title":"Intestinal infection due to Clostridium difficile"},{"icd_code":"5772","icd_version":9,"title":"Cyst and pseudocyst of pancreas"},{"icd_code":"7907","icd_version":9,"title":"Bacteremia"},{"icd_code":"1179","icd_version":9,"title":"Other and unspecified mycoses


In [10]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    train_dataset = dataset, eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 1e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = SEED, report_to = "none", output_dir = "outputs",
        gradient_checkpointing = True,
        save_strategy = "steps", save_steps = 100,
    ),
)
trainer = train_on_responses_only(trainer)
trainer_stats = trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1992 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Unsloth: Auto-detected instruction_part = '<|turn>user\n' and response_part = '<|turn>model\n'


Map:   0%|          | 0/1992 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,992 | Num Epochs = 1 | Total steps = 249
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 25,337,856 of 5,148,515,872 (0.49% trained)


Step,Training Loss
10,2.485585
20,1.544907
30,1.212872
40,0.955137
50,0.823319
60,0.763622
70,0.699283
80,0.653867
90,0.644407
100,0.610311


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-249/tokenizer_config.json.


In [11]:
model.save_pretrained("gemma_mimic_lora")
tokenizer.save_pretrained("gemma_mimic_lora")
from google.colab import drive; drive.mount('/content/drive')
!cp -r gemma_mimic_lora /content/drive/MyDrive/
print("saved: gemma_mimic_lora (local + Google Drive)")

Unsloth: Restored added_tokens_decoder metadata in gemma_mimic_lora/tokenizer_config.json.


Mounted at /content/drive
saved: gemma_mimic_lora (local + Google Drive)


In [12]:
N_SYNTH = 10

FastModel.for_inference(model)
torch.manual_seed(SEED)

KEYS = set(full_records[0][1].keys())

def is_num(x): return isinstance(x, (int, float)) and not isinstance(x, bool)
def str_or_none(x): return x is None or isinstance(x, str)

def valid(rec):
    if not isinstance(rec, dict) or set(rec.keys()) != KEYS: return False
    if not all(isinstance(rec[k], str) for k in ("gender", "race", "admission_type")): return False
    if not all(str_or_none(rec[k]) for k in ("insurance", "language", "marital_status", "admission_location", "discharge_location")): return False
    if not all(is_num(rec[k]) for k in ("age", "los_days", "num_diagnoses", "num_procedures", "num_medications")): return False
    if rec["died_in_hospital"] not in (0, 1): return False
    ic = rec["icu"]
    if ic is not None:
        if not isinstance(ic, dict) or set(ic.keys()) != {"first_careunit", "last_careunit", "icu_los_days"}: return False
        if not (isinstance(ic["first_careunit"], str) and isinstance(ic["last_careunit"], str) and is_num(ic["icu_los_days"])): return False
    for k in ("diagnoses", "procedures"):
        if not isinstance(rec[k], list): return False
        for d in rec[k]:
            if not (isinstance(d, dict) and set(d.keys()) == {"icd_code", "icd_version", "title"}
                    and isinstance(d["icd_code"], str) and is_num(d["icd_version"]) and isinstance(d["title"], str)): return False
    if not isinstance(rec["labs"], list): return False
    for l in rec["labs"]:
        if not (isinstance(l, dict) and set(l.keys()) == {"test", "value", "unit", "flag"}
                and isinstance(l["test"], str) and is_num(l["value"]) and str_or_none(l["unit"]) and isinstance(l["flag"], str)): return False
    if not (isinstance(rec["medications"], list) and all(isinstance(x, str) for x in rec["medications"])): return False
    return True

def parse(txt):
    txt = re.sub(r"^```(?:json)?\s*|\s*```$", "", txt.strip())
    try: rec = json.loads(txt)
    except Exception: return None
    return rec if valid(rec) else None

def generate_records(n_total, per_call=8, temperature=1.0, top_p=0.95, top_k=64):
    msgs = [{"role": "user", "content": PROMPT}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                     tokenize=True, return_dict=True).to("cuda")
    plen = inputs["input_ids"].shape[1]
    out, bad, tries = [], 0, 0
    while len(out) < n_total and tries < 6 * n_total // 4 + 20:
        tries += 1
        try:
            gen = model.generate(**inputs, max_new_tokens=GEN_TOKENS, do_sample=True,
                                 temperature=temperature, top_p=top_p, top_k=top_k,
                                 num_return_sequences=per_call)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); per_call = max(1, per_call // 2)
            print("OOM -> per_call =", per_call); continue
        for g in gen:
            rec = parse(tok.decode(g[plen:], skip_special_tokens=True))
            if rec is None: bad += 1
            else: out.append(rec)
        print(f"  progress: {len(out)}/{n_total}", end="\r")
    print(f"\nvalid={len(out)} invalid={bad} validity rate={len(out)/max(len(out)+bad,1):.1%}")
    return out[:n_total]

synth = generate_records(N_SYNTH)
print(len(synth), "synthetic records")
for s in synth[:2]:
    print(json.dumps(s, ensure_ascii=False)[:1200], "\n")


  progress: 16/10
valid=16 invalid=0 validity rate=100.0%
10 synthetic records
{"gender": "M", "age": 69, "race": "WHITE", "insurance": "Medicare", "language": "ENGLISH", "marital_status": "MARRIED", "admission_type": "ELECTIVE", "admission_location": "PHYSICIAN REFERRAL", "discharge_location": "HOME HEALTH CARE", "los_days": 16.9, "died_in_hospital": 0, "icu": null, "num_diagnoses": 32, "num_procedures": 5, "num_medications": 36, "diagnoses": [{"icd_code": "18620", "icd_version": 9, "title": "Chronic kidney disease, stage 5 or end stage, unspecified"}, {"icd_code": "2761", "icd_version": 9, "title": "Hypernatremia"}, {"icd_code": "5990", "icd_version": 9, "title": "Urinary tract infection, site not specified"}, {"icd_code": "5271", "icd_version": 9, "title": "Acute posthemorrhagic anemia"}, {"icd_code": "51820", "icd_version": 9, "title": "Chronic pulmonary edema"}], "procedures": [{"icd_code": "3907", "icd_version": 9, "title": "Hemodialysis"}], "labs": [{"test": "Sodium", "value": 1

In [13]:
from scipy.stats import ks_2samp
from collections import Counter

real_views = [make_view(r) for _, r in full_records]
real_full  = [r for _, r in full_records]

def feats(r):
    ic = r["icu"] or {}
    return {"gender": r["gender"], "race": r["race"], "insurance": r["insurance"], "language": r["language"],
            "marital_status": r["marital_status"], "admission_type": r["admission_type"],
            "admission_location": r["admission_location"], "discharge_location": r["discharge_location"],
            "died": r["died_in_hospital"], "has_icu": int(r["icu"] is not None),
            "icu_careunit": ic.get("first_careunit"),
            "age": r["age"], "los_days": r["los_days"], "icu_los": ic.get("icu_los_days"),
            "num_dx": r["num_diagnoses"], "num_proc": r["num_procedures"], "num_meds": r["num_medications"],
            "n_labs": len(r["labs"]),
            "primary_dx": r["diagnoses"][0]["icd_code"][:3] if r["diagnoses"] else None}

R = pd.DataFrame([feats(r) for r in real_views]); S = pd.DataFrame([feats(r) for r in synth])
print(f"real={len(R)} synthetic={len(S)}   (chhota synthetic sample: percentages sirf indicative hain)")

print("\n== 1. CATEGORICAL: kya synthetic labels real labels me se hain? ==")
for c in ["gender", "race", "insurance", "language", "marital_status", "admission_type", "admission_location",
          "discharge_location", "icu_careunit"]:
    real_set = set(R[c].astype(str))
    novel = (~S[c].astype(str).isin(real_set)).mean()
    print(f"  {c:20s} novel labels={novel:.0%} | synth: {dict(S[c].astype(str).value_counts())}")
print("  has_icu : real=%.2f synth=%.2f | died: real=%.2f synth=%.2f" % (R.has_icu.mean(), S.has_icu.mean(), R.died.mean(), S.died.mean()))

print("\n== 2. NUMERIC RANGES ==")
for c in ["age", "los_days", "icu_los", "num_dx", "num_proc", "num_meds", "n_labs"]:
    r_, s_ = R[c].dropna().astype(float), S[c].dropna().astype(float)
    if len(s_) == 0: print(f"  {c}: no values"); continue
    print(f"  {c:10s} real mean={r_.mean():7.2f} med={r_.median():6.2f} [{r_.min():.1f}, {r_.max():.1f}] | "
          f"synth mean={s_.mean():7.2f} med={s_.median():6.2f} [{s_.min():.1f}, {s_.max():.1f}] | "
          f"in-range={s_.between(r_.min(), r_.max()).mean():.0%}")

print("\n== 3. LAB VALUES (real mean/std vs synthetic) ==")
def lab_vals(records):
    d = {}
    for r in records:
        for l in r["labs"]: d.setdefault(l["test"], []).append(l["value"])
    return d
real_lab, syn_lab = lab_vals(real_full), lab_vals(synth)
units = {t: Counter(l["unit"] for r in real_full for l in r["labs"] if l["test"] == t).most_common(1)[0][0] for t in real_lab}
tot_in, tot_n = 0, 0
for t in TOP_LABS:
    s_ = np.array(syn_lab.get(t, []), dtype=float); r_ = np.array(real_lab.get(t, []), dtype=float)
    if len(s_) == 0: continue
    inr = (s_ >= r_.min()) & (s_ <= r_.max()); tot_in += inr.sum(); tot_n += len(s_)
    print(f"  {t:22s} n={len(s_):2d} real {r_.mean():8.2f}±{r_.std():6.2f} | synth {s_.mean():8.2f} | in-range={inr.mean():.0%}")
all_labs = [l for r in synth for l in r["labs"]]
print(f"  OVERALL lab values in real range: {tot_in/max(tot_n,1):.1%} ({tot_n} values)")
print(f"  lab test names valid: {np.mean([l['test'] in LAB_ORDER for l in all_labs]):.1%} | "
      f"unit matches real: {np.mean([l['unit'] == units.get(l['test']) for l in all_labs]):.1%} | "
      f"abnormal share real={np.mean([l['flag']=='abnormal' for r in real_full for l in r['labs']]):.3f} "
      f"synth={np.mean([l['flag']=='abnormal' for l in all_labs]):.3f}")

print("\n== 4. ICD CODES / CONSISTENCY ==")
dx_title = {(str(c), int(v)): t for c, v, t in zip(d_dx.icd_code, d_dx.icd_version, d_dx.long_title)}
px_title = {(str(c), int(v)): t for c, v, t in zip(d_px.icd_code, d_px.icd_version, d_px.long_title)}
for name, ref in [("diagnoses", dx_title), ("procedures", px_title)]:
    items = [(d["icd_code"], int(d["icd_version"]), d["title"]) for r in synth for d in r[name]]
    if items:
        print(f"  {name}: code exists={np.mean([(c, v) in ref for c, v, _ in items]):.1%} | "
              f"code-title pair correct={np.mean([ref.get((c, v)) == t for c, v, t in items]):.1%} ({len(items)} items)")
print("  list length consistent with num_* fields: "
      f"{np.mean([len(r['diagnoses']) == min(DX_CAP, r['num_diagnoses']) and len(r['procedures']) == min(PX_CAP, r['num_procedures']) for r in synth]):.1%} (real = 100%)")
print("  died_in_hospital vs discharge_location=='DIED' consistent: "
      f"{np.mean([(r['died_in_hospital'] == 1) == (r['discharge_location'] == 'DIED') for r in synth]):.1%} | "
      f"real: {np.mean([(r['died_in_hospital'] == 1) == (r['discharge_location'] == 'DIED') for r in real_full]):.1%}")

print("\n== 5. MEMORIZATION ==")
key = lambda r: (r["age"], r["los_days"], tuple(d["icd_code"] for d in r["diagnoses"]))
train_keys, hold_keys = {key(r) for r in train_base}, {key(r) for r in hold_base}
sj = [json.dumps(r, sort_keys=True) for r in synth]
print(f"  copies of TRAIN admissions  : {sum(key(r) in train_keys for r in synth)}/{len(synth)}")
print(f"  copies of HOLDOUT admissions: {sum(key(r) in hold_keys for r in synth)}/{len(synth)}")
print(f"  duplicates inside synthetic : {len(sj) - len(set(sj))}/{len(synth)}")

real=275 synthetic=10   (chhota synthetic sample: percentages sirf indicative hain)

== 1. CATEGORICAL: kya synthetic labels real labels me se hain? ==
  gender               novel labels=0% | synth: {'M': np.int64(6), 'F': np.int64(4)}
  race                 novel labels=0% | synth: {'WHITE': np.int64(10)}
  insurance            novel labels=0% | synth: {'Other': np.int64(6), 'Medicare': np.int64(4)}
  language             novel labels=0% | synth: {'ENGLISH': np.int64(10)}
  marital_status       novel labels=0% | synth: {'WIDOWED': np.int64(4), 'MARRIED': np.int64(3), 'DIVORCED': np.int64(2), 'SINGLE': np.int64(1)}
  admission_type       novel labels=0% | synth: {'EW EMER.': np.int64(6), 'ELECTIVE': np.int64(2), 'URGENT': np.int64(1), 'DIRECT EMER.': np.int64(1)}
  admission_location   novel labels=0% | synth: {'EMERGENCY ROOM': np.int64(6), 'PHYSICIAN REFERRAL': np.int64(2), 'TRANSFER FROM HOSPITAL': np.int64(1), 'CLINIC REFERRAL': np.int64(1)}
  discharge_location   novel labels=0% 

In [14]:
rows = [{**feats(r), "json": json.dumps(r, ensure_ascii=False)} for r in synth]
pd.DataFrame(rows).to_csv("synthetic_mimic.csv", index=False)
with open("synthetic_mimic.json", "w", encoding="utf-8") as f:
    json.dump(synth, f, ensure_ascii=False, indent=2)
from google.colab import files
files.download("synthetic_mimic.csv"); files.download("synthetic_mimic.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
N_SYNTH   = 30
GEN_TEMP  = 0.7
GEN_TOP_P = 0.9

FastModel.for_inference(model)
torch.manual_seed(SEED)

KEYS = set(full_records[0][1].keys())

def is_num(x): return isinstance(x, (int, float)) and not isinstance(x, bool)
def str_or_none(x): return x is None or isinstance(x, str)

def valid(rec):
    if not isinstance(rec, dict) or set(rec.keys()) != KEYS: return False
    if not all(isinstance(rec[k], str) for k in ("gender", "race", "admission_type")): return False
    if not all(str_or_none(rec[k]) for k in ("insurance", "language", "marital_status", "admission_location", "discharge_location")): return False
    if not all(is_num(rec[k]) for k in ("age", "los_days", "num_diagnoses", "num_procedures", "num_medications")): return False
    if rec["died_in_hospital"] not in (0, 1): return False
    ic = rec["icu"]
    if ic is not None:
        if not isinstance(ic, dict) or set(ic.keys()) != {"first_careunit", "last_careunit", "icu_los_days"}: return False
        if not (isinstance(ic["first_careunit"], str) and isinstance(ic["last_careunit"], str) and is_num(ic["icu_los_days"])): return False
    for k in ("diagnoses", "procedures"):
        if not isinstance(rec[k], list): return False
        for d in rec[k]:
            if not (isinstance(d, dict) and set(d.keys()) == {"icd_code", "icd_version", "title"}
                    and isinstance(d["icd_code"], str) and is_num(d["icd_version"]) and isinstance(d["title"], str)): return False
    if not isinstance(rec["labs"], list): return False
    for l in rec["labs"]:
        if not (isinstance(l, dict) and set(l.keys()) == {"test", "value", "unit", "flag"}
                and isinstance(l["test"], str) and is_num(l["value"]) and str_or_none(l["unit"]) and isinstance(l["flag"], str)): return False
    if not (isinstance(rec["medications"], list) and all(isinstance(x, str) for x in rec["medications"])): return False
    return True

def parse(txt):
    txt = re.sub(r"^```(?:json)?\s*|\s*```$", "", txt.strip())
    try: rec = json.loads(txt)
    except Exception: return None
    return rec if valid(rec) else None

def generate_records(n_total, per_call=8, temperature=GEN_TEMP, top_p=GEN_TOP_P, top_k=64):
    msgs = [{"role": "user", "content": PROMPT}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt",
                                     tokenize=True, return_dict=True).to("cuda")
    plen = inputs["input_ids"].shape[1]
    out, bad, tries = [], 0, 0
    while len(out) < n_total and tries < 6 * n_total // 4 + 20:
        tries += 1
        try:
            gen = model.generate(**inputs, max_new_tokens=GEN_TOKENS, do_sample=True,
                                 temperature=temperature, top_p=top_p, top_k=top_k,
                                 num_return_sequences=per_call)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); per_call = max(1, per_call // 2)
            print("OOM -> per_call =", per_call); continue
        for g in gen:
            rec = parse(tok.decode(g[plen:], skip_special_tokens=True))
            if rec is None: bad += 1
            else: out.append(rec)
        print(f"  progress: {min(len(out), n_total)}/{n_total}", end="\r")
    print(f"\nvalid={len(out)} invalid={bad} validity rate={len(out)/max(len(out)+bad,1):.1%}")
    return out[:n_total]

synth = generate_records(N_SYNTH)
print(len(synth), "synthetic records | temperature =", GEN_TEMP)
for s in synth[:2]:
    print(json.dumps(s, ensure_ascii=False)[:1200], "\n")

  progress: 30/30
valid=32 invalid=0 validity rate=100.0%
30 synthetic records | temperature = 0.7
{"gender": "M", "age": 69, "race": "WHITE", "insurance": "Medicare", "language": "ENGLISH", "marital_status": "MARRIED", "admission_type": "ELECTIVE", "admission_location": "PHYSICIAN REFERRAL", "discharge_location": "HOME HEALTH CARE", "los_days": 16.9, "died_in_hospital": 0, "icu": null, "num_diagnoses": 32, "num_procedures": 0, "num_medications": 30, "diagnoses": [{"icd_code": "18622", "icd_version": 9, "title": "Chronic kidney disease, stage 5 or end stage, unspecified"}, {"icd_code": "2761", "icd_version": 9, "title": "Hypernatremia"}, {"icd_code": "28549", "icd_version": 9, "title": "Obesity, unspecified"}, {"icd_code": "2769", "icd_version": 9, "title": "Pure hyperlipidemia"}, {"icd_code": "2762", "icd_version": 9, "title": "Mixed hyperlipidemia"}], "procedures": [], "labs": [{"test": "Chloride", "value": 100.0, "unit": "mEq/L", "flag": "normal"}, {"test": "Platelet Count", "value"

In [16]:
from scipy.stats import ks_2samp
from collections import Counter

real_views = [make_view(r) for _, r in full_records]
real_full  = [r for _, r in full_records]

def feats(r):
    ic = r["icu"] or {}
    return {"gender": r["gender"], "race": r["race"], "insurance": r["insurance"], "language": r["language"],
            "marital_status": r["marital_status"], "admission_type": r["admission_type"],
            "admission_location": r["admission_location"], "discharge_location": r["discharge_location"],
            "died": r["died_in_hospital"], "has_icu": int(r["icu"] is not None),
            "icu_careunit": ic.get("first_careunit"),
            "age": r["age"], "los_days": r["los_days"], "icu_los": ic.get("icu_los_days"),
            "num_dx": r["num_diagnoses"], "num_proc": r["num_procedures"], "num_meds": r["num_medications"],
            "n_labs": len(r["labs"]),
            "primary_dx": r["diagnoses"][0]["icd_code"][:3] if r["diagnoses"] else None}

def tvd(a, b):
    pa, pb = a.astype(str).value_counts(normalize=True), b.astype(str).value_counts(normalize=True)
    idx = pa.index.union(pb.index)
    return 0.5 * (pa.reindex(idx, fill_value=0) - pb.reindex(idx, fill_value=0)).abs().sum()

R = pd.DataFrame([feats(r) for r in real_views]); S = pd.DataFrame([feats(r) for r in synth])
print(f"real={len(R)} synthetic={len(S)}   (30 records: TVD/KS indicative hain, final proof nahi)")

print("\n== 1. CATEGORICAL (TVD: 0 = identical | novel = real me na milne wale labels) ==")
for c in ["gender", "race", "insurance", "language", "marital_status", "admission_type", "admission_location",
          "discharge_location", "icu_careunit"]:
    novel = (~S[c].astype(str).isin(set(R[c].astype(str)))).mean()
    print(f"  {c:20s} TVD={tvd(R[c], S[c]):.3f} novel={novel:.0%} | synth top: {dict(S[c].astype(str).value_counts().head(3))}")
print("  has_icu: real=%.2f synth=%.2f | died: real=%.2f synth=%.2f | race WHITE: real=%.2f synth=%.2f" % (
    R.has_icu.mean(), S.has_icu.mean(), R.died.mean(), S.died.mean(),
    (R.race == "WHITE").mean(), (S.race == "WHITE").mean()))

print("\n== 2. NUMERIC (KS: 0 = identical | in-range = real min..max ke andar) ==")
for c in ["age", "los_days", "icu_los", "num_dx", "num_proc", "num_meds", "n_labs"]:
    r_, s_ = R[c].dropna().astype(float), S[c].dropna().astype(float)
    if len(s_) < 3: print(f"  {c}: kam values"); continue
    ks = ks_2samp(r_, s_).statistic if len(s_) >= 5 else float("nan")
    print(f"  {c:10s} real mean={r_.mean():7.2f} med={r_.median():6.2f} [{r_.min():.1f}, {r_.max():.1f}] | "
          f"synth mean={s_.mean():7.2f} med={s_.median():6.2f} [{s_.min():.1f}, {s_.max():.1f}] | "
          f"KS={ks:.3f} in-range={s_.between(r_.min(), r_.max()).mean():.0%}")

print("\n== 3. LAB VALUES ==")
def lab_vals(records):
    d = {}
    for r in records:
        for l in r["labs"]: d.setdefault(l["test"], []).append(l["value"])
    return d
real_lab, syn_lab = lab_vals(real_full), lab_vals(synth)
units = {t: Counter(l["unit"] for r in real_full for l in r["labs"] if l["test"] == t).most_common(1)[0][0] for t in real_lab}
tot_in, tot_n = 0, 0
for t in TOP_LABS:
    s_ = np.array(syn_lab.get(t, []), dtype=float); r_ = np.array(real_lab.get(t, []), dtype=float)
    if len(s_) == 0: continue
    inr = (s_ >= r_.min()) & (s_ <= r_.max()); tot_in += inr.sum(); tot_n += len(s_)
    ks = f"{ks_2samp(r_, s_).statistic:.3f}" if len(s_) >= 8 else " n/a "
    print(f"  {t:22s} n={len(s_):2d} real {r_.mean():8.2f}±{r_.std():7.2f} | synth {s_.mean():8.2f} | KS={ks} in-range={inr.mean():.0%}")
all_labs = [l for r in synth for l in r["labs"]]
print(f"  OVERALL lab values in real range: {tot_in/max(tot_n,1):.1%} ({tot_n} values)")
print(f"  lab test names valid: {np.mean([l['test'] in LAB_ORDER for l in all_labs]):.1%} | "
      f"unit matches real: {np.mean([l['unit'] == units.get(l['test']) for l in all_labs]):.1%} | "
      f"abnormal share real={np.mean([l['flag']=='abnormal' for r in real_full for l in r['labs']]):.3f} "
      f"synth={np.mean([l['flag']=='abnormal' for l in all_labs]):.3f}")

print("\n== 4. CORRELATION (core variables) ==")
core = ["age", "los_days", "num_dx", "num_meds", "died"]
diff = np.nanmean(np.abs(R[core].astype(float).corr().values - S[core].astype(float).corr().values))
print(f"  mean abs diff of correlation matrices: {diff:.3f}")

print("\n== 5. ICD CODES / CONSISTENCY ==")
dx_title = {(str(c), int(v)): t for c, v, t in zip(d_dx.icd_code, d_dx.icd_version, d_dx.long_title)}
px_title = {(str(c), int(v)): t for c, v, t in zip(d_px.icd_code, d_px.icd_version, d_px.long_title)}
for name, ref in [("diagnoses", dx_title), ("procedures", px_title)]:
    items = [(d["icd_code"], int(d["icd_version"]), d["title"]) for r in synth for d in r[name]]
    if items:
        print(f"  {name}: code exists={np.mean([(c, v) in ref for c, v, _ in items]):.1%} | "
              f"code-title pair correct={np.mean([ref.get((c, v)) == t for c, v, t in items]):.1%} ({len(items)} items)")
print("  list length consistent with num_* fields: "
      f"{np.mean([len(r['diagnoses']) == min(DX_CAP, r['num_diagnoses']) and len(r['procedures']) == min(PX_CAP, r['num_procedures']) for r in synth]):.1%} (real = 100%)")
print("  died_in_hospital vs discharge_location=='DIED' consistent: "
      f"{np.mean([(r['died_in_hospital'] == 1) == (r['discharge_location'] == 'DIED') for r in synth]):.1%} (real = 100%)")
print("  ICD mismatch examples (pehle 5):")
shown = 0
for r in synth:
    for d in r["diagnoses"]:
        key_ = (d["icd_code"], int(d["icd_version"]))
        if dx_title.get(key_) != d["title"] and shown < 5:
            print(f"    {d['icd_code']} v{int(d['icd_version'])} | synth title: {d['title'][:50]} | dictionary says: {str(dx_title.get(key_, 'CODE NOT IN DICTIONARY'))[:50]}")
            shown += 1

print("\n== 6. MEMORIZATION ==")
key = lambda r: (r["age"], r["los_days"], tuple(d["icd_code"] for d in r["diagnoses"]))
train_keys, hold_keys = {key(r) for r in train_base}, {key(r) for r in hold_base}
sj = [json.dumps(r, sort_keys=True) for r in synth]
print(f"  copies of TRAIN admissions  : {sum(key(r) in train_keys for r in synth)}/{len(synth)}")
print(f"  copies of HOLDOUT admissions: {sum(key(r) in hold_keys for r in synth)}/{len(synth)}")
print(f"  duplicates inside synthetic : {len(sj) - len(set(sj))}/{len(synth)}")

real=275 synthetic=30   (30 records: TVD/KS indicative hain, final proof nahi)

== 1. CATEGORICAL (TVD: 0 = identical | novel = real me na milne wale labels) ==
  gender               TVD=0.050 novel=0% | synth top: {'M': np.int64(17), 'F': np.int64(13)}
  race                 TVD=0.248 novel=0% | synth top: {'WHITE': np.int64(26), 'BLACK/AFRICAN AMERICAN': np.int64(4)}
  insurance            TVD=0.292 novel=0% | synth top: {'Other': np.int64(25), 'Medicare': np.int64(5)}
  language             TVD=0.073 novel=0% | synth top: {'ENGLISH': np.int64(30)}
  marital_status       TVD=0.208 novel=0% | synth top: {'MARRIED': np.int64(13), 'SINGLE': np.int64(8), 'WIDOWED': np.int64(7)}
  admission_type       TVD=0.422 novel=0% | synth top: {'EW EMER.': np.int64(24), 'OBSERVATION ADMIT': np.int64(4), 'ELECTIVE': np.int64(1)}
  admission_location   TVD=0.413 novel=0% | synth top: {'EMERGENCY ROOM': np.int64(27), 'PHYSICIAN REFERRAL': np.int64(3)}
  discharge_location   TVD=0.395 novel=0% | synth 

In [17]:
rows = [{**feats(r), "json": json.dumps(r, ensure_ascii=False)} for r in synth]
pd.DataFrame(rows).to_csv("synthetic_mimic_30.csv", index=False)
with open("synthetic_mimic_30.json", "w", encoding="utf-8") as f:
    json.dump(synth, f, ensure_ascii=False, indent=2)
from google.colab import files
files.download("synthetic_mimic_30.csv"); files.download("synthetic_mimic_30.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>